# 01 — Data Preparation

**Lineage-Aware Hierarchical Deep Learning for Imbalanced Multi-Class Classification of Peripheral Blood Cells**
Arthiga Karthigesu

This notebook executes objectives 2 and 3 of the proposal: *analyse the MLL23 dataset and characterise its
18 classes and imbalance profile*, and *preprocess the single-cell images and define a two-level label
hierarchy that reflects haematopoietic lineage*.

It is idempotent — re-running skips work already on disk. Reusable logic lives in `src/`; this notebook
drives it and produces the figures for the write-up.

| Stage | Output |
|---|---|
| 1. Acquire | 41,621 TIFFs in `D:\MLL23\raw\<class>\` |
| 2. Manifest | `interim/manifest.csv` — one row per image with `y1`, `y2` |
| 3. Imbalance profile | Figure: per-class distribution |
| 4. Stratified split | `interim/splits.csv` — deterministic 70/15/15 |
| 5. Transforms | 288→224 bilinear, ImageNet norm, class-conditional augmentation |
| 6. Dataset | `MLL23Dataset` yielding `(image, y1, y2)` |

In [10]:
%load_ext autoreload
%autoreload 2

import sys, pathlib
if ".." not in sys.path:
    sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np
import pandas as pd
import torch

from src import config, manifest, splits, transforms as T, viz
from src.hierarchy import CLASSES, LINEAGES, NUM_FINE_CLASSES, TOTAL_EXPECTED_IMAGES

viz.apply_style()
pd.set_option("display.max_rows", 40)

print(f"data root      {config.DATA_ROOT}")
print(f"input size     {config.SOURCE_IMAGE_SIZE} -> {config.IMAGE_SIZE} px (all backbones)")
print(f"split          {config.TRAIN_FRAC:.0%}/{config.VAL_FRAC:.0%}/{config.TEST_FRAC:.0%}  seed={config.SPLIT_SEED}")
print(f"classes        {NUM_FINE_CLASSES} fine / {len(LINEAGES)} lineages, {TOTAL_EXPECTED_IMAGES:,} images expected")
print(f"torch          {torch.__version__}  cuda={torch.cuda.is_available()}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
data root      D:\MLL23
input size     288 -> 224 px (all backbones)
split          70%/15%/15%  seed=42
classes        18 fine / 3 lineages, 41,621 images expected
torch          2.6.0+cu124  cuda=True


## 1. Acquire the dataset

MLL23 is published on Zenodo ([10.5281/zenodo.14277609](https://doi.org/10.5281/zenodo.14277609), CC BY 4.0)
as 18 per-class ZIP archives totalling 9.1 GB.

`download_all` fetches one archive at a time, verifies its MD5 against the published checksum, extracts it,
then deletes the archive before moving to the next — so peak disk usage is the extracted corpus plus one
archive, not all 18 at once. Classes already extracted are skipped, so re-running this cell is cheap.

In [11]:
from src.download import download_all

counts = download_all()
print(f"\ntotal images on disk: {sum(counts.values()):,}")

SSLError: [ASN1: NOT_ENOUGH_DATA] not enough data (_ssl.c:4040)

## 2. Build and verify the manifest

The manifest is the single source of truth for what is on disk — one row per image carrying both hierarchy
levels. Everything downstream reads it rather than re-walking the filesystem, so splits, class weights and
datasets can never disagree about the corpus.

The verification step compares observed per-class counts against the counts published in the MLL23 data
descriptor. A mismatch means a truncated download or a mis-mapped archive, either of which would silently
corrupt every experiment.

In [12]:
mdf = manifest.build_manifest()
report = manifest.verify_manifest(mdf)

display(report.style.hide(axis="index").format({"expected": "{:,}", "observed": "{:,}"}))

if report["ok"].all():
    print(f"\nAll 18 classes match the published counts. Total {len(mdf):,} images.")
    manifest.save(mdf)
    print(f"manifest -> {config.MANIFEST_PATH}")
else:
    bad = report.loc[~report["ok"], ["class_name", "expected", "observed", "delta"]]
    raise SystemExit(f"Manifest does not match published counts:\n{bad.to_string(index=False)}")

FileNotFoundError: missing class folder: D:\MLL23\raw\myeloblast

## 3. Imbalance profile

This is the central challenge the dissertation addresses. The distribution is *not* an artefact to be
corrected — it reflects the true frequency of these cell types in peripheral blood, and preserving it is
what makes the evaluation clinically meaningful.

In [8]:
summary = manifest.imbalance_summary(mdf)
for k, v in summary.items():
    print(f"{k:18s} {v}")

NameError: name 'mdf' is not defined

In [9]:
fig = viz.class_distribution(
    mdf["y2"].value_counts(),
    title="MLL23 class distribution — 41,621 single-cell images across 18 classes",
)
fig.savefig(config.ARTIFACT_DIR / "class_distribution.png")

NameError: name 'mdf' is not defined

The linear panel shows why plain accuracy is unusable here: a classifier that ignored the seven rarest
classes entirely would still score well. The log panel makes those classes legible. Reactive lymphocytes
(33 images) against myeloblasts (8,606) is the 260:1 ratio quoted in the proposal.

In [3]:
lineage_tbl = (
    mdf.groupby("lineage", sort=False)
       .agg(classes=("y2", "nunique"), images=("path", "count"))
       .reindex(list(LINEAGES))
)
lineage_tbl["share"] = (lineage_tbl["images"] / len(mdf)).map("{:.1%}".format)
print("Level 1 — haematopoietic lineage\n")
print(lineage_tbl.to_string())

NameError: name 'mdf' is not defined

## 4. Deterministic stratified split

Stratification is on the fine label `y2`, which stratifies lineage for free since every fine class belongs
to exactly one lineage. The seed is fixed so all four experiment configurations (flat baseline, full
hierarchical model, and the two ablations) train and evaluate on identical partitions — without that, the
ablation comparison measures split noise rather than the component being ablated.

The binding constraint is reactive lymphocytes at 33 images: a 70/15/15 split leaves roughly 23/5/5.
`check_split` asserts every class is non-empty in every split rather than assuming it.

In [ ]:
sdf = splits.make_splits(mdf)
problems = splits.check_split(sdf)

if problems:
    for p in problems:
        print("PROBLEM:", p)
    raise SystemExit("split failed validation")

print("Split validation passed — every class present in every split, no leakage.\n")
print(sdf["split"].value_counts(normalize=True).reindex(list(splits.SPLIT_NAMES)).map("{:.3f}".format).to_string())

splits.save(sdf)
print(f"\nsplits -> {config.SPLIT_PATH}")

In [ ]:
split_tbl = splits.split_summary(sdf)
display(split_tbl)

fig = viz.split_composition(split_tbl)
fig.savefig(config.ARTIFACT_DIR / "split_composition.png")

## 5. Preprocessing and augmentation

Per the proposal:

1. **Resize** — 288×288 → 224×224, bilinear, antialiased. The same resolution for *every* backbone
   including the ViT, so the backbone comparison varies architecture alone.
2. **Scale and normalise** — to `[0,1]`, then ImageNet mean/std, to match the pretrained weights.
3. **Augment (training split only)** — random flips, rotation, mild brightness/contrast jitter, with an
   intensified regime on minority classes.

Rotation is applied at the native 288 px resolution before the downsample, so the image is resampled once
rather than twice. Rotating a square frame empties the corners; these are filled with white, matching the
bright background of a Pappenheim-stained smear, so the fill introduces no colour the network would not
otherwise encounter.

In [ ]:
minority = T.minority_class_names()
print(f"Intensified augmentation applies to {len(minority)} classes at or below "
      f"{config.MINORITY_THRESHOLD:,} images:\n")
for name in minority:
    print(f"  - {name}")

print(f"\n  majority regime:  rotation +/-{config.MAX_ROTATION_DEGREES:.0f} deg, jitter {config.JITTER_STRENGTH}")
print(f"  minority regime:  rotation +/-{min(config.MAX_ROTATION_DEGREES * config.MINORITY_AUG_MULTIPLIER, 180):.0f} deg, "
      f"jitter {config.JITTER_STRENGTH * config.MINORITY_AUG_MULTIPLIER:.1f}")

In [ ]:
from PIL import Image

# One image from the rarest class, shown deterministically and then augmented.
rare = mdf[mdf.class_name == "Reactive lymphocytes"].iloc[0]
src_img = Image.open(rare.path).convert("RGB")
print(f"{rare.class_name}: {src_img.size[0]}x{src_img.size[1]} {Image.open(rare.path).mode} -> "
      f"{config.IMAGE_SIZE}x{config.IMAGE_SIZE}")

torch.manual_seed(0)
aug = T.train_transform(minority=True)
imgs = [T.denormalise(T.eval_transform()(src_img)).numpy()] + \
       [T.denormalise(aug(src_img)).numpy() for _ in range(11)]
labels = ["eval (deterministic)"] + [f"augmented {i+1}" for i in range(11)]

fig = viz.sample_grid(imgs, labels, ncols=6,
                      title="Minority-class augmentation — one reactive lymphocyte, 11 draws")
fig.savefig(config.ARTIFACT_DIR / "augmentation_examples.png")

In [ ]:
# One example per class, deterministic pipeline.
ev = T.eval_transform()
picks = [mdf[mdf.y2 == c.idx].iloc[0] for c in CLASSES]
imgs = [T.denormalise(ev(Image.open(r.path).convert("RGB"))).numpy() for r in picks]
labels = [f"{c.name}\n(n={c.expected_count:,})" for c in CLASSES]

fig = viz.sample_grid(imgs, labels, ncols=6, title="One example per class after preprocessing")
fig.savefig(config.ARTIFACT_DIR / "class_examples.png")

## 6. Dataset and DataLoader

`MLL23Dataset` yields `(image, y1, y2)`. The flat baseline simply ignores `y1`, so all four experiment
configurations share one dataset implementation and cannot drift apart in how they read data.

In [ ]:
from torch.utils.data import DataLoader
from src.dataset import from_splits

ds = {s: from_splits(sdf, s) for s in splits.SPLIT_NAMES}
for s, d in ds.items():
    print(f"{s:6s} {len(d):>6,} images   augmentation={'on' if d.train else 'off'}")

loader = DataLoader(ds["train"], batch_size=32, shuffle=True, num_workers=0)
x, y1, y2 = next(iter(loader))

print(f"\nbatch images  {tuple(x.shape)}  {x.dtype}")
print(f"batch y1      {tuple(y1.shape)}  values {sorted(set(y1.tolist()))}")
print(f"batch y2      {tuple(y2.shape)}  range [{y2.min().item()}, {y2.max().item()}]")
print(f"normalised    mean {x.mean():+.3f}  std {x.std():.3f}   (≈0/≈1 confirms ImageNet normalisation)")

In [ ]:
# The hierarchy must be internally consistent: y1 has to be recoverable from y2.
from src.hierarchy import FINE_TO_LINEAGE

lut = torch.tensor(FINE_TO_LINEAGE)
assert torch.equal(lut[y2], y1), "y1 is not recoverable from y2 — hierarchy mapping is broken"

full = torch.tensor(sdf["y2"].to_numpy())
assert torch.equal(lut[full], torch.tensor(sdf["y1"].to_numpy())), "manifest y1/y2 disagree"

assert x.shape[1:] == (3, config.IMAGE_SIZE, config.IMAGE_SIZE)
assert len(sdf) == TOTAL_EXPECTED_IMAGES

print("All consistency checks passed.")
print(f"  y1 recoverable from y2 for all {len(sdf):,} rows")
print(f"  every image {config.IMAGE_SIZE}x{config.IMAGE_SIZE}, 3 channels")

In [ ]:
# Class counts of the training split feed the class-balanced loss in notebook 02.
counts = ds["train"].class_counts
print("Training-split counts per fine class (input to the class-balanced loss):\n")
for c in CLASSES:
    n = int(counts[c.idx])
    print(f"  [{c.idx:2d}] {c.name:36s} {n:>6,}  {c.lineage}")
print(f"\n  imbalance ratio within train: {counts.max() / counts.min():.1f}:1")

## Summary

Preprocessing is complete and verified:

- **41,621 images** downloaded, MD5-verified, and matched class-by-class against the published counts.
- **Two-level hierarchy** encoded, with `y1` provably recoverable from `y2`, and fine-class indices ordered
  along the myeloid maturation continuum so confusion matrices place adjacent stages together.
- **Deterministic 70/15/15 stratified split** saved, with every class — including the 33-image reactive
  lymphocyte class — represented in all three partitions.
- **Transform pipeline** fixed at 224×224 for all backbones, with class-conditional augmentation strength.

Artifacts written to `artifacts/` for the write-up. Splits and manifest are on disk, so notebook 02 can
build the model without re-running any of this.

**Next (notebook 02):** shared backbone with the two heads, the class-balanced hierarchical loss, and the
flat baseline — trained against these fixed splits.